In [1]:
import os
import polars as pl
import pandas as pd
from tqdm.notebook import tqdm

In [2]:
# genhpf
# GENDER: event type : code_type,  value : code_part_1
# MEDS_BIRTH: event type : code_type
# MEDS_DEATH: event type : code_type
# ED_REGISTRATION:  event type : code_type 
# ED_OUT: event type : code_type
# eGFR: event type : code_type, value : numeric_value_original
# BMI: event type : code_type, value : numeric_value_original
# BMI (kg/m2): event type : code_type,  value : numeric_value_original
# Weight: event type : code_type, value : numeric_value_original
# Weight (Lbs): event type : code_type, value : numeric_value_original
# Height: event type : code_type, value : numeric_value_original
# Height (Inches): event type : code_type,  value : numeric_value_original
# Blood Pressure: event type : code_type,  value : numeric_value_original
# Blood Pressure Lying: event type : code_type,  value : numeric_value_original
# Blood Pressure Sitting: event type : code_type,  value : numeric_value_original
# Blood Pressure Standing: event type : code_type,  value : numeric_value_original
# Blood Pressure Standing (1 min): event type : code_type,  value : numeric_value_original
# Blood Pressure Standing (3 mins): event type : code_type,  value : numeric_value_original
# PROCEDURE_ICD: event type : code_type, version : code_part_2, value : code_part_3
# DIAGNOSIS_ICD: event type : code_type, version : code_part_2, value : code_part_3
# HOSPITAL_ADMISSION: event type: code_type,  admission location : code_part_2
# HOSPITAL_DISCHARGE: event type : code_type, discharge location : code_part_1
# ICU_ADMISSION: event type : code_type, care unit : code_part_1
# ICU_DISCHARGE: event type : code_type, care unit : code_part_1
# TRANSFER_TO: event type : code_type, transfer location: code_part_2
# TIME-GAP: event type : code_type, value : text value
# DRG: event type : code_type, drg category : code_part_1, drg description : code_part_3
# SUBJECT_WEIGHT_AT_INFUSION: event type : code_type, value : numeric_value_original, unit : code_part_1
# MEDICATION_START: event type : code_type,  medication : code_part_2
# MEDICATION_STOP: evevnt type : code_type, medication : code_part_2
# MEDICATION: event type : code_type, medication : code_part_1
# ICU_CHART_EVENT: event type : code_type, chart event name : code_part_1_mapped, value : numeric_value_original unit : code_part_2
# ICU_PROCEDURE_START: event type : code_type, procedure name : code_part_2_mapped
# ICU_PROCEDURE_END: event type : code_type, procedure name : code_part_2_mapped
# LAB_SPECIMEN: event type : code_type, lab test name : code_part_2_mapped 
# LAB_RESULT: event type : code_type, lab test name : code_part_2_mapped, value : numeric_value_original, unit : code_part_3
# ICU_SUBJECT_FLUID_OUTPUT: event type : code_type, output fluid name : code_part_1_mapped,  value : numeric_value_original, unit : code_part_2
# ICU_INFUSION_START: event type : code_type, infused medication name : code_part_1_mapped, value : numeric_value_original, unit : unit
# ICU_INFUSION_END: event type : code_type, infused medication name : code_part_1_mapped value: numeric_value_original, unit : unit

In [3]:
# descemb

# GENDER: code_type code_part_1
# MEDS_BIRTH: code_type
# MEDS_DEATH: code_type
# ED_REGISTRATION: code_type 
# ED_OUT: code_type
# eGFR: code_type numeric_value_original  
# BMI: code_type numeric_value_original
# BMI (kg/m2): code_type  numeric_value_original
# Weight: code_type numeric_value_original
# Weight (Lbs): code_type numeric_value_original
# Height: code_type numeric_value_original
# Height (Inches): code_type  numeric_value_original
# Blood Pressure: code_type  numeric_value_original
# Blood Pressure Lying: code_type numeric_value_original
# Blood Pressure Sitting: code_type  numeric_value_original
# Blood Pressure Standing: code_type numeric_value_original
# Blood Pressure Standing (1 min): code_type  numeric_value_original
# Blood Pressure Standing (3 mins): code_type  numeric_value_original
# PROCEDURE_ICD: code_type code_part_2 code_part_3
# DIAGNOSIS_ICD: code_type code_part_2 code_part_3
# HOSPITAL_ADMISSION: code_type  code_part_2
# HOSPITAL_DISCHARGE: code_type code_part_1
# ICU_ADMISSION: code_type code_part_1
# ICU_DISCHARGE: code_type code_part_1
# TRANSFER_TO: code_type  code_part_2
# TIME-GAP: code_type text value
# DRG: code_type code_part_1 code_part_3
# SUBJECT_WEIGHT_AT_INFUSION: code_type numeric_value_original code_part_1
# MEDICATION_START: code_type code_part_2
# MEDICATION_STOP: code_type code_part_2
# MEDICATION: code_type code_part_1
# ICU_CHART_EVENT: code_type code_part_1_mapped numeric_value_original code_part_2
# ICU_PROCEDURE_START: code_type code_part_2_mapped
# ICU_PROCEDURE_END: code_type code_part_2_mapped
# LAB_SPECIMEN: code_type code_part_2_mapped 
# LAB_RESULT: code_type code_part_2_mapped numeric_value_original code_part_3
# ICU_SUBJECT_FLUID_OUTPUT: code_type code_part_1_mapped numeric_value_original code_part_2
# ICU_INFUSION_START: code_type code_part_1_mapped numeric_value_original unit
# ICU_INFUSION_END: code_type code_part_1_mapped numeric_value_original unit

In [4]:
def build_event_mapping_table(shard_path):

    df = pl.read_parquet(
        shard_path,
    )

    df = (
        df
        .with_columns(
            pl.col("code")
            .str.split("//")
            .alias("_code_parts")
        )
        .with_columns(
            [
                pl.col("_code_parts")
                .list.get(i, null_on_oob=True)
                .alias(f"code_part_{i}")
                for i in range(5)
            ]
        )
        .drop("_code_parts")
    )

    return df

In [5]:
def build_itemid_mapping_dict(
    lab_mapping_path,
    icu_mapping_path,
):

    lab_df = pd.read_csv(lab_mapping_path)

    icu_df = pd.read_csv(icu_mapping_path)


    lab_df["label"] = (
        lab_df["label"]
        .fillna("")
        .astype(str)
    )

    icu_df["label"] = (
        icu_df["label"]
        .fillna("")
        .astype(str)
    )


    lab_itemid_to_label = dict(
        zip(
            lab_df["itemid"].astype(str),
            lab_df["label"]
        )
    )


    icu_itemid_to_label = dict(
        zip(
            icu_df["itemid"].astype(str),
            icu_df["label"]
        )
    )


    return (
        lab_itemid_to_label,
        icu_itemid_to_label,
    )

In [6]:
import polars as pl


def map_code_parts(
    df,
    lab_dict,
    icu_dict,
):

    lab_types = [
        "LAB_SPECIMEN",
        "LAB_RESULT",
    ]

    icu_part2_types = [
        "ICU_PROCEDURE_END",
        "ICU_PROCEDURE_START",
    ]

    icu_part1_types = [
        "ICU_INFUSION_START",
        "ICU_INFUSION_END",
        "ICU_SUBJECT_FLUID_OUTPUT",
        "ICU_CHART_EVENT",
    ]


    df = df.with_columns(
        [
            pl.col("code_part_1")
            .cast(pl.Utf8)
            .alias("code_part_1_mapped"),

            pl.col("code_part_2")
            .cast(pl.Utf8)
            .alias("code_part_2_mapped"),
        ]
    )


    df = df.with_columns(
        [

            # ---------------------------------------
            # code_part_1 mappings
            # ICU_CHART_EVENT
            # ICU_INFUSION_START/END
            # ICU_SUBJECT_FLUID_OUTPUT
            # ---------------------------------------
            pl.when(
                pl.col("code_type").is_in(icu_part1_types)
            )
            .then(
                pl.col("code_part_1")
                .replace(icu_dict)
            )
            .otherwise(
                pl.col("code_part_1_mapped")
            )
            .alias("code_part_1_mapped"),


            # ---------------------------------------
            # code_part_2 mappings
            # LAB
            # ICU_PROCEDURES
            # ---------------------------------------
            pl.when(
                pl.col("code_type").is_in(lab_types)
            )
            .then(
                pl.col("code_part_2")
                .replace(lab_dict)
            )
            .when(
                pl.col("code_type").is_in(icu_part2_types)
            )
            .then(
                pl.col("code_part_2")
                .replace(icu_dict)
            )
            .otherwise(
                pl.col("code_part_2_mapped")
            )
            .alias("code_part_2_mapped"),

        ]
    )


    return df

In [7]:
import polars as pl

def dsva_number(x):
    """
    Digit-Split Value Aggregation with max 3 decimals:
    12.34567  -> '1 2 . 3 5'
    5         -> '5'
    5.1       -> '5 . 1'
    """
    if x is None:
        return None
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None

    # round to 3 decimal places
    v = round(v, 3)

    # format to 3 decimals, then strip trailing zeros and dot
    s = f"{v:.3f}".rstrip("0").rstrip(".")

    # DSVA: split into individual characters separated by spaces
    return " ".join(list(s))


def dsva_expr(col: pl.Expr) -> pl.Expr:
    return (
        col.cast(pl.Float64)
           .map_elements(dsva_number, return_dtype=pl.Utf8)
    )


In [8]:
import polars as pl


def add_descemb_v2(df: pl.DataFrame) -> pl.DataFrame:

    ct = pl.col("code_type")

    gender_norm = (
        pl.when(pl.col("code_part_1").str.to_lowercase() == "m")
        .then(pl.lit("male"))
        .when(pl.col("code_part_1").str.to_lowercase() == "f")
        .then(pl.lit("female"))
        .otherwise(pl.col("code_part_1"))
    )


    descemb = (

        # -----------------------------
        # Simple code_type only
        # -----------------------------
        pl.when(
            ct.is_in(
                [
                    "MEDS_BIRTH",
                    "MEDS_DEATH",
                    "ED_REGISTRATION",
                    "ED_OUT",
                ]
            )
        )
        .then(ct)


        # -----------------------------
        # Gender
        # -----------------------------
        .when(ct == "GENDER")
        .then(
            pl.concat_str(
                [
                    ct,
                    gender_norm,
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )


        # -----------------------------
        # Numeric measurements
        # -----------------------------
        .when(
            ct.is_in(
                [
                    "eGFR",
                    "BMI",
                    "BMI (kg/m2)",
                    "Weight",
                    "Weight (Lbs)",
                    "Height",
                    "Height (Inches)",
                    "Blood Pressure",
                    "Blood Pressure Lying",
                    "Blood Pressure Sitting",
                    "Blood Pressure Standing",
                    "Blood Pressure Standing (1 min)",
                    "Blood Pressure Standing (3 mins)",
                ]
            )
        )
        .then(
            pl.concat_str(
                [
                    ct,
#                     dsva_expr(pl.col("numeric_value_original")),
                    pl.col("numeric_value_original"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )


        # -----------------------------
        # ICD / Procedure
        # -----------------------------
        .when(ct == "DIAGNOSIS_ICD")
        .then(
            pl.concat_str(
                [
                    pl.lit("diagnosis icd"),
                    pl.col("code_part_2"),
                    pl.col("code_part_3"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )

        .when(ct == "PROCEDURE_ICD")
        .then(
            pl.concat_str(
                [
                    pl.lit("procedure icd"),
                    pl.col("code_part_2"),
                    pl.col("code_part_3"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )


        # -----------------------------
        # Admission / discharge
        # -----------------------------
        .when(ct == "HOSPITAL_ADMISSION")
        .then(
            pl.concat_str(
                [
                    ct,
                    pl.col("code_part_2"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )

        .when(
            ct.is_in(
                [
                    "HOSPITAL_DISCHARGE",
                    "ICU_ADMISSION",
                    "ICU_DISCHARGE",
                ]
            )
        )
        .then(
            pl.concat_str(
                [
                    ct,
                    pl.col("code_part_1"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )


        # -----------------------------
        # Transfer
        # -----------------------------
        .when(ct == "TRANSFER_TO")
        .then(
            pl.concat_str(
                [
                    ct,
                    pl.col("code_part_2"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )


        # -----------------------------
        # Time gap
        # -----------------------------
        .when(ct == "TIME-GAP")
        .then(
            pl.concat_str(
                [
                    ct,
                    pl.col("text_value"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )


        # -----------------------------
        # DRG
        # -----------------------------
        .when(ct == "DRG")
        .then(
            pl.concat_str(
                [
                    ct,
                    pl.col("code_part_1"),
                    pl.col("code_part_3"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )


        # -----------------------------
        # Weight during infusion
        # -----------------------------
        .when(ct == "SUBJECT_WEIGHT_AT_INFUSION")
        .then(
            pl.concat_str(
                [
                    ct,
#                     dsva_expr(pl.col("numeric_value_original")),
                    pl.col("numeric_value_original"),
                    pl.col("code_part_1"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )


        # -----------------------------
        # Medication
        # -----------------------------
        .when(ct == "MEDICATION_START")
        .then(
            pl.concat_str(
                [
                    ct,
                    pl.col("code_part_2"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )

        .when(ct == "MEDICATION_STOP")
        .then(
            pl.concat_str(
                [
                    ct,
                    pl.col("code_part_2"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )

        .when(ct == "MEDICATION")
        .then(
            pl.concat_str(
                [
                    ct,
                    pl.col("code_part_1"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )


        # -----------------------------
        # ICU mapped events
        # -----------------------------
        .when(ct == "ICU_CHART_EVENT")
        .then(
            pl.concat_str(
                [
                    ct,
                    pl.col("code_part_1_mapped"),
#                     dsva_expr(pl.col("numeric_value_original")),
                    pl.col("numeric_value_original"),
                    pl.col("code_part_2"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )

        .when(
            ct.is_in(
                [
                    "ICU_PROCEDURE_START",
                    "ICU_PROCEDURE_END",
                ]
            )
        )
        .then(
            pl.concat_str(
                [
                    ct,
                    pl.col("code_part_2_mapped"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )

        .when(ct == "LAB_SPECIMEN")
        .then(
            pl.concat_str(
                [
                    ct,
                    pl.col("code_part_2_mapped"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )

        .when(ct == "LAB_RESULT")
        .then(
            pl.concat_str(
                [
                    ct,
                    pl.col("code_part_2_mapped"),
#                     dsva_expr(pl.col("numeric_value_original")),
                    pl.col("numeric_value_original"),
                    pl.col("code_part_3"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )


        .when(ct == "ICU_SUBJECT_FLUID_OUTPUT")
        .then(
            pl.concat_str(
                [
                    ct,
                    pl.col("code_part_1_mapped"),
#                     dsva_expr(pl.col("numeric_value_original")),
                    pl.col("numeric_value_original"),
                    pl.col("code_part_2"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )


        .when(
            ct.is_in(
                [
                    "ICU_INFUSION_START",
                    "ICU_INFUSION_END",
                ]
            )
        )
        .then(
            pl.concat_str(
                [
                    ct,
                    pl.col("code_part_1_mapped"),
#                     dsva_expr(pl.col("numeric_value_original")),
                    pl.col("numeric_value_original"),
                    pl.col("unit"),
                ],
                separator=" ",
                ignore_nulls=True,
            )
        )


        .otherwise(None)
    )


    return (
        df
        .with_columns(
            descemb.alias("descemb")
        )
        .with_columns(
            pl.col("descemb")
            .str.replace_all(r"[_\-]+", " ")
            .str.to_lowercase()
            .str.strip_chars()
        )
    )

In [9]:
import polars as pl


def add_genhpf_v2(df: pl.DataFrame) -> pl.DataFrame:

    ct_clean = (
        pl.col("code_type")
        .str.replace_all(r"[_\-]+", " ")
        .str.to_lowercase()
        .str.strip_chars()
    )


    gender_norm = (
        pl.when(pl.col("code_part_1").str.to_lowercase() == "m")
        .then(pl.lit("male"))
        .when(pl.col("code_part_1").str.to_lowercase() == "f")
        .then(pl.lit("female"))
        .otherwise(pl.col("code_part_1"))
    )


    def event_text(*cols):

        return pl.concat_str(
            [
                pl.concat_str(
                    [
                        pl.lit("event type: "),
                        ct_clean,
                    ],
                    separator="",
                )
            ]
            + list(cols),
            separator=", ",
            ignore_nulls=True,
        )


    genhpf_expr = (

        # -----------------------------------------
        # Simple events
        # -----------------------------------------
        pl.when(
            pl.col("code_type").is_in(
                [
                    "MEDS_BIRTH",
                    "MEDS_DEATH",
                    "ED_REGISTRATION",
                    "ED_OUT",
                ]
            )
        )
        .then(
            event_text()
        )


        # -----------------------------------------
        # Gender
        # -----------------------------------------
        .when(pl.col("code_type") == "GENDER")
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("value: "),
                        gender_norm,
                    ],
                    separator="",
                )
            )
        )


        # -----------------------------------------
        # Measurements
        # -----------------------------------------
        .when(
            pl.col("code_type").is_in(
                [
                    "eGFR",
                    "BMI",
                    "BMI (kg/m2)",
                    "Weight",
                    "Weight (Lbs)",
                    "Height",
                    "Height (Inches)",
                    "Blood Pressure",
                    "Blood Pressure Lying",
                    "Blood Pressure Sitting",
                    "Blood Pressure Standing",
                    "Blood Pressure Standing (1 min)",
                    "Blood Pressure Standing (3 mins)",
                ]
            )
        )
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("value: "),
#                         dsva_expr(pl.col("numeric_value_original")),
                        pl.col("numeric_value_original"),
                    ],
                    separator="",
                )
            )
        )


        # -----------------------------------------
        # ICD
        # -----------------------------------------
        .when(pl.col("code_type") == "DIAGNOSIS_ICD")
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("version: "),
                        pl.col("code_part_2"),
                    ],
                    separator="",
                ),
                pl.concat_str(
                    [
                        pl.lit("value: "),
                        pl.col("code_part_3"),
                    ],
                    separator="",
                ),
            )
        )

        .when(pl.col("code_type") == "PROCEDURE_ICD")
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("version: "),
                        pl.col("code_part_2"),
                    ],
                    separator="",
                ),
                pl.concat_str(
                    [
                        pl.lit("value: "),
                        pl.col("code_part_3"),
                    ],
                    separator="",
                ),
            )
        )


        # -----------------------------------------
        # Admission/discharge/location
        # -----------------------------------------
        .when(pl.col("code_type") == "HOSPITAL_ADMISSION")
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("admission location: "),
                        pl.col("code_part_2"),
                    ],
                    separator="",
                )
            )
        )

        .when(pl.col("code_type") == "HOSPITAL_DISCHARGE")
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("discharge location: "),
                        pl.col("code_part_1"),
                    ],
                    separator="",
                )
            )
        )

        .when(
            pl.col("code_type").is_in(
                [
                    "ICU_ADMISSION",
                    "ICU_DISCHARGE",
                ]
            )
        )
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("care unit: "),
                        pl.col("code_part_1"),
                    ],
                    separator="",
                )
            )
        )

        .when(pl.col("code_type") == "TRANSFER_TO")
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("transfer location: "),
                        pl.col("code_part_2"),
                    ],
                    separator="",
                )
            )
        )


        # -----------------------------------------
        # DRG
        # -----------------------------------------
        .when(pl.col("code_type") == "DRG")
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("drg category: "),
                        pl.col("code_part_1"),
                    ],
                    separator="",
                ),
                pl.concat_str(
                    [
                        pl.lit("drg description: "),
                        pl.col("code_part_3"),
                    ],
                    separator="",
                ),
            )
        )


        # -----------------------------------------
        # Time gap
        # -----------------------------------------
        .when(pl.col("code_type") == "TIME-GAP")
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("value: "),
                        pl.col("text_value"),
                    ],
                    separator="",
                )
            )
        )


        # -----------------------------------------
        # Subject weight infusion
        # -----------------------------------------
        .when(pl.col("code_type") == "SUBJECT_WEIGHT_AT_INFUSION")
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("value: "),
#                         dsva_expr(pl.col("numeric_value_original")),
                        pl.col("numeric_value_original"),
                    ],
                    separator="",
                ),
                pl.concat_str(
                    [
                        pl.lit("unit: "),
                        pl.col("code_part_1"),
                    ],
                    separator="",
                ),
            )
        )


        # -----------------------------------------
        # Medication
        # -----------------------------------------
        .when(
            pl.col("code_type").is_in(
                [
                    "MEDICATION",
                    "MEDICATION_START",
                    "MEDICATION_STOP",
                ]
            )
        )
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("medication: "),
                        pl.coalesce(
                            [
                                pl.col("code_part_1"),
                                pl.col("code_part_2"),
                            ]
                        ),
                    ],
                    separator="",
                )
            )
        )


        # -----------------------------------------
        # ICU chart
        # -----------------------------------------
        .when(pl.col("code_type") == "ICU_CHART_EVENT")
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("chart event name: "),
                        pl.col("code_part_1_mapped"),
                    ],
                    separator="",
                ),
                pl.concat_str(
                    [
                        pl.lit("value: "),
#                         dsva_expr(pl.col("numeric_value_original")),
                        pl.col("numeric_value_original"),
                    ],
                    separator="",
                ),
                pl.concat_str(
                    [
                        pl.lit("unit: "),
                        pl.col("code_part_2"),
                    ],
                    separator="",
                ),
            )
        )


        # -----------------------------------------
        # ICU procedures
        # -----------------------------------------
        .when(
            pl.col("code_type").is_in(
                [
                    "ICU_PROCEDURE_START",
                    "ICU_PROCEDURE_END",
                ]
            )
        )
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("procedure name: "),
                        pl.col("code_part_2_mapped"),
                    ],
                    separator="",
                )
            )
        )


        # -----------------------------------------
        # Labs
        # -----------------------------------------
        .when(pl.col("code_type") == "LAB_SPECIMEN")
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("lab test name: "),
                        pl.col("code_part_2_mapped"),
                    ],
                    separator="",
                )
            )
        )


        .when(pl.col("code_type") == "LAB_RESULT")
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("lab test name: "),
                        pl.col("code_part_2_mapped"),
                    ],
                    separator="",
                ),
                pl.concat_str(
                    [
                        pl.lit("value: "),
#                         dsva_expr(pl.col("numeric_value_original")),
                        pl.col("numeric_value_original"),
                        
                    ],
                    separator="",
                ),
                pl.concat_str(
                    [
                        pl.lit("unit: "),
                        pl.col("code_part_3"),
                    ],
                    separator="",
                ),
            )
        )


        # -----------------------------------------
        # Fluid output
        # -----------------------------------------
        .when(pl.col("code_type") == "ICU_SUBJECT_FLUID_OUTPUT")
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("output fluid name: "),
                        pl.col("code_part_1_mapped"),
                    ],
                    separator="",
                ),
                pl.concat_str(
                    [
                        pl.lit("value: "),
#                         dsva_expr(pl.col("numeric_value_original")),
                        pl.col("numeric_value_original"),
                    ],
                    separator="",
                ),
                pl.concat_str(
                    [
                        pl.lit("unit: "),
                        pl.col("code_part_2"),
                    ],
                    separator="",
                ),
            )
        )


        # -----------------------------------------
        # Infusion
        # -----------------------------------------
        .when(
            pl.col("code_type").is_in(
                [
                    "ICU_INFUSION_START",
                    "ICU_INFUSION_END",
                ]
            )
        )
        .then(
            event_text(
                pl.concat_str(
                    [
                        pl.lit("infused medication name: "),
                        pl.col("code_part_1_mapped"),
                    ],
                    separator="",
                ),
                pl.concat_str(
                    [
                        pl.lit("value: "),
#                         dsva_expr(pl.col("numeric_value_original")),
                        pl.col("numeric_value_original"),
                    ],
                    separator="",
                ),
                pl.concat_str(
                    [
                        pl.lit("unit: "),
                        pl.col("unit"),
                    ],
                    separator="",
                ),
            )
        )


        .otherwise(None)
    )


    return (
        df
        .with_columns(
            genhpf_expr.alias("genhpf")
        )
        .with_columns(
            pl.col("genhpf")
            .str.replace_all(r"[_\-]+", " ")
            .str.to_lowercase()
            .str.strip_chars()
        )
    )

In [10]:
def sample_descemb_by_code_type(
    df,
    code_types,
    n=1,
):
    samples = []

    for ct in code_types:

        sample = (
            df
            .filter(
                pl.col("code_type") == ct
            )
            .select(
                [
                    "code_type",
                    "code",
                    "descemb",
                ]
            )
            .head(n)
        )

        samples.append(sample)

    return pl.concat(
        samples,
        how="vertical"
    )

In [11]:
# data_path = os.path.join('..','data')
# descemb_genhpf_data_path = os.path.join(data_path,'llm-data', 'data',)
# os.makedirs(descemb_genhpf_data_path,exist_ok=True)
# splits = ['train', 'tuning', 'held_out']

# normalzied_path = os.path.join('..','data', 'meds_normalized')
# for split in splits:
#     os.makedirs(os.path.join(descemb_genhpf_data_path,split),exist_ok=True)

# for split in splits:
    
#     for file in tqdm(os.listdir(os.path.join(normalzied_path,'data',split))):
#         shard = build_event_mapping_table(os.path.join(normalzied_path,'data',split,file))
#         os.path.join('..','resources','old-resources','mimic-mapping','d_labitems.csv')   
                                          
#         lab_dict, icu_dict = build_itemid_mapping_dict(
#             lab_mapping_path=os.path.join('..','resources','old-resources','mimic-mapping','d_labitems.csv'),
#             icu_mapping_path=os.path.join('..','resources','old-resources','mimic-mapping','d_items.csv'),
#         )
                    
#         shard = shard.with_columns(
#             [
#                 pl.col("code_part_1").cast(pl.Utf8),
#                 pl.col("code_part_2").cast(pl.Utf8),
#             ]
#         )
                                          
#         shard = map_code_parts(
#             shard,
#             lab_dict,
#             icu_dict,
#         )
#         shard = add_descemb_v2(shard)
#         shard = add_genhpf_v2(shard)
#         shard.write_parquet(os.path.join(descemb_genhpf_data_path,split,file))

In [12]:
# limits = {
#     'within24_query': {512:  ['w24_start_512',  'w24_end_512' ]},
#     'within48_query': {512:  ['w48_start_512',  'w48_end_512' ]},
#     'within_stay_query': {512:  ['wStay_start_512',  'wStay_end_512' ]},
#     }

In [13]:
# pl.Config.set_tbl_rows(600)
# data_idx = pl.read_parquet('../resources/downstream_index.parquet')
# x = 0
# subject_id = data_idx[x]['subject_id'][0]
# shard = data_idx[x]['shard'][0]
# split = data_idx[x]['split'][0]
# start = data_idx[x]['wStay_start_1024'][0]
# end = data_idx[x]['wStay_end_1024'][0]
# a = pl.read_parquet(f'../data/meds_normalized/data/{split}/{shard}').filter(pl.col('subject_id') == subject_id)
# print(a[start:end].shape[0])

In [14]:
# from datasets import Dataset, Features, Sequence, Value
# data_idx = pl.read_parquet('../resources/downstream_index.parquet')

# def gen():
#     for i in range(len(data_idx)):
#         subject_id = data_idx[i]['subject_id'][0]
#         icustay_id = data_idx[i]['icustay_id'][0]
#         shard = data_idx[i]['shard'][0]
#         split = data_idx[i]['split'][0]
#         w24_start = data_idx[i]['w24_start_512'][0]
#         w24_end = data_idx[i]['w24_end_512'][0]
        
#         w48_start = data_idx[i]['w48_start_512'][0]
#         w48_end = data_idx[i]['w48_end_512'][0]
        
#         wstay_start = data_idx[i]['wStay_start_512'][0]
#         wstay_end = data_idx[i]['wStay_end_512'][0]
        
#         file = pl.read_parquet(os.path.join('..','data','descemb_genhpf_data','data',split,shard))
#         seq = file.filter(pl.col('subject_id') == subject_id)
        
        

#         yield {
#             "subject_id": subject_id,
#             "icustay_id": icustay_id,

            
#             "within24_descemb": seq[w24_start:w24_end,:]['descemb'].to_list(),
#             "within48_descemb": seq[w48_start:w48_end,:]['descemb'].to_list(),
#             "within_stay_descemb": seq[wstay_start:wstay_end,:]['descemb'].to_list(),
            
#             "within24_genhpf": seq[w24_start:w24_end,:]['genhpf'].to_list(),
#             "within48_genhpf": seq[w48_start:w48_end,:]['genhpf'].to_list(),
#             "within_stay_genhpf": seq[wstay_start:wstay_end,:]['genhpf'].to_list(),
            
#             "within24_time": seq[w24_start:w24_end, :]["time"].to_list(),
#             "within48_time": seq[w48_start:w48_end, :]["time"].to_list(),
#             "within_stay_time": seq[wstay_start:wstay_end, :]["time"].to_list(),

#             "within24_time_diff": seq[w24_start:w24_end, :]["time_diff"].to_list(),
#             "within48_time_diff": seq[w48_start:w48_end, :]["time_diff"].to_list(),
#             "within_stay_time_diff": seq[wstay_start:wstay_end, :]["time_diff"].to_list(),
            
            
#             "within24_remed": seq[0:w24_end,:]['genhpf'].to_list(),
#             "within24_remed_time": seq[0:w24_end, :]["time"].to_list(),
#             "within24_remed_time_diff": seq[0:w24_end, :]["time_diff"].to_list(),
            
#             "within48_remed": seq[0:w48_end,:]['genhpf'].to_list(),
#             "within48_remed_time": seq[0:w48_end, :]["time"].to_list(),
#             "within48_remed_time_diff": seq[0:w48_end, :]["time_diff"].to_list(),
            
#             "within_stay_remed": seq[0:wstay_end,:]['genhpf'].to_list(),
#             "within_stay_remed_time": seq[0:wstay_end, :]["time"].to_list(),
#             "within_stay_remed_time_diff": seq[0:wstay_end, :]["time_diff"].to_list(),
            
#             }

# features = Features({
#     "subject_id": Value("int64"),
#     "icustay_id": Value("int64"),


#     "within24_descemb": Sequence(Value("string")),
#     "within48_descemb": Sequence(Value("string")),
#     "within_stay_descemb": Sequence(Value("string")),

#     "within24_genhpf": Sequence(Value("string")),
#     "within48_genhpf": Sequence(Value("string")),
#     "within_stay_genhpf": Sequence(Value("string")),


#     "within24_time": Sequence(Value("timestamp[us]")),
#     "within48_time": Sequence(Value("timestamp[us]")),
#     "within_stay_time": Sequence(Value("timestamp[us]")),

#     "within24_time_diff": Sequence(Value("float32")),
#     "within48_time_diff": Sequence(Value("float32")),
#     "within_stay_time_diff": Sequence(Value("float32")),

#     "within24_remed": Sequence(Value("string")),
#     "within48_remed": Sequence(Value("string")),
#     "within_stay_remed": Sequence(Value("string")),

#     "within24_remed_time": Sequence(Value("timestamp[us]")),
#     "within48_remed_time": Sequence(Value("timestamp[us]")),
#     "within_stay_remed_time": Sequence(Value("timestamp[us]")),

#     "within24_remed_time_diff": Sequence(Value("float32")),
#     "within48_remed_time_diff": Sequence(Value("float32")),
#     "within_stay_remed_time_diff": Sequence(Value("float32")),
# })

# ds_arrow = Dataset.from_generator(
#     gen,
#     features=features,
#     writer_batch_size=1000  # tune for shard sizes
# )

# # write Arrow shards to disk (memory-mappable)
# ds_arrow.save_to_disk(os.path.join('..','data','desc_gen_dataset'))

# # optional: set PyTorch formatting
# # ds_arrow.set_format(type="torch")



# # # later / in training script:
# # from datasets import load_from_disk
# # train = load_from_disk("ehr_arrow_dataset")
# # train.set_format(type="torch")

# Big note (BAAAM)
1. LLM data does not consider dvsa for numerical value 
2. The code for this part is commented out for this part and uses the original values

In [15]:
limits = {
    'within24_query': {1024:  ['w24_start_1024',  'w24_end_1024' ]},
    'within48_query': {1024:  ['w48_start_1024',  'w48_end_1024' ]},
    'within_stay_query': {1024:  ['wStay_start_1024',  'wStay_end_1024']},
    }

In [16]:
# from datasets import Dataset, Features, Sequence, Value
# data_idx = pl.read_parquet('../resources/downstream_index.parquet')

# def gen():
#     for i in range(len(data_idx)):
#         subject_id = data_idx[i]['subject_id'][0]
#         icustay_id = data_idx[i]['icustay_id'][0]
#         shard = data_idx[i]['shard'][0]
#         split = data_idx[i]['split'][0]
#         w24_start = data_idx[i]['w24_start_1024'][0]
#         w24_end = data_idx[i]['w24_end_1024'][0]
        
#         w48_start = data_idx[i]['w48_start_1024'][0]
#         w48_end = data_idx[i]['w48_end_1024'][0]
        
#         wstay_start = data_idx[i]['wStay_start_1024'][0]
#         wstay_end = data_idx[i]['wStay_end_1024'][0]
        
#         file = pl.read_parquet(os.path.join('..','data','llm-data','data',split,shard))
#         seq = file.filter(pl.col('subject_id') == subject_id)
        
        

#         yield {
#             "subject_id": subject_id,
#             "icustay_id": icustay_id,

            
#             "within24_descemb": seq[w24_start:w24_end,:]['descemb'].to_list(),
#             "within48_descemb": seq[w48_start:w48_end,:]['descemb'].to_list(),
#             "within_stay_descemb": seq[wstay_start:wstay_end,:]['descemb'].to_list(),
            
#             "within24_genhpf": seq[w24_start:w24_end,:]['genhpf'].to_list(),
#             "within48_genhpf": seq[w48_start:w48_end,:]['genhpf'].to_list(),
#             "within_stay_genhpf": seq[wstay_start:wstay_end,:]['genhpf'].to_list(),
            
#             "within24_time": seq[w24_start:w24_end, :]["time"].to_list(),
#             "within48_time": seq[w48_start:w48_end, :]["time"].to_list(),
#             "within_stay_time": seq[wstay_start:wstay_end, :]["time"].to_list(),

#             "within24_time_diff": seq[w24_start:w24_end, :]["time_diff"].to_list(),
#             "within48_time_diff": seq[w48_start:w48_end, :]["time_diff"].to_list(),
#             "within_stay_time_diff": seq[wstay_start:wstay_end, :]["time_diff"].to_list(),
            
            
#             "within24_remed": seq[0:w24_end,:]['genhpf'].to_list(),
#             "within24_remed_time": seq[0:w24_end, :]["time"].to_list(),
#             "within24_remed_time_diff": seq[0:w24_end, :]["time_diff"].to_list(),
            
#             "within48_remed": seq[0:w48_end,:]['genhpf'].to_list(),
#             "within48_remed_time": seq[0:w48_end, :]["time"].to_list(),
#             "within48_remed_time_diff": seq[0:w48_end, :]["time_diff"].to_list(),
            
#             "within_stay_remed": seq[0:wstay_end,:]['genhpf'].to_list(),
#             "within_stay_remed_time": seq[0:wstay_end, :]["time"].to_list(),
#             "within_stay_remed_time_diff": seq[0:wstay_end, :]["time_diff"].to_list(),
            
#             }

# features = Features({
#     "subject_id": Value("int64"),
#     "icustay_id": Value("int64"),


#     "within24_descemb": Sequence(Value("string")),
#     "within48_descemb": Sequence(Value("string")),
#     "within_stay_descemb": Sequence(Value("string")),

#     "within24_genhpf": Sequence(Value("string")),
#     "within48_genhpf": Sequence(Value("string")),
#     "within_stay_genhpf": Sequence(Value("string")),


#     "within24_time": Sequence(Value("timestamp[us]")),
#     "within48_time": Sequence(Value("timestamp[us]")),
#     "within_stay_time": Sequence(Value("timestamp[us]")),

#     "within24_time_diff": Sequence(Value("float32")),
#     "within48_time_diff": Sequence(Value("float32")),
#     "within_stay_time_diff": Sequence(Value("float32")),

#     "within24_remed": Sequence(Value("string")),
#     "within48_remed": Sequence(Value("string")),
#     "within_stay_remed": Sequence(Value("string")),

#     "within24_remed_time": Sequence(Value("timestamp[us]")),
#     "within48_remed_time": Sequence(Value("timestamp[us]")),
#     "within_stay_remed_time": Sequence(Value("timestamp[us]")),

#     "within24_remed_time_diff": Sequence(Value("float32")),
#     "within48_remed_time_diff": Sequence(Value("float32")),
#     "within_stay_remed_time_diff": Sequence(Value("float32")),
# })

# ds_arrow = Dataset.from_generator(
#     gen,
#     features=features,
#     writer_batch_size=1000  # tune for shard sizes
# )

# # write Arrow shards to disk (memory-mappable)
# ds_arrow.save_to_disk(os.path.join('..','data','llm-dataset'))

# # optional: set PyTorch formatting
# # ds_arrow.set_format(type="torch")



# # # later / in training script:
# # from datasets import load_from_disk
# # train = load_from_disk("ehr_arrow_dataset")
# # train.set_format(type="torch")

In [17]:
import polars as pl

In [18]:
idx = pl.read_parquet('../resources/downstream_index.parquet')
idx.group_by("split").len()

split,len
str,u32
"""tuning""",5986
"""train""",49080
"""held_out""",6097


In [19]:
idx.group_by("split").agg([
    pl.col("y_mort").count(),
    pl.col("y_los_7").count(),
    pl.col("y_mort_12mo").count(),
    pl.col("y_icu_readmit_30").count(),
])

split,y_mort,y_los_7,y_mort_12mo,y_icu_readmit_30
str,u32,u32,u32,u32
"""tuning""",5986,5986,5986,5986
"""held_out""",6097,6097,6097,6097
"""train""",49080,49080,49080,49080


In [20]:
idx.group_by("split").agg([
    pl.len().alias("n"),

    pl.col("y_mort").sum().alias("y_mort_pos"),
    (pl.col("y_mort").mean() * 100).alias("y_mort_prev_%"),

    pl.col("y_los_7").sum().alias("y_los_7_pos"),
    (pl.col("y_los_7").mean() * 100).alias("y_los_7_prev_%"),

    pl.col("y_mort_12mo").sum().alias("y_mort_12mo_pos"),
    (pl.col("y_mort_12mo").mean() * 100).alias("y_mort_12mo_prev_%"),

    pl.col("y_icu_readmit_30").sum().alias("y_icu_readmit_30_pos"),
    (pl.col("y_icu_readmit_30").mean() * 100).alias("y_icu_readmit_30_prev_%"),
])

split,n,y_mort_pos,y_mort_prev_%,y_los_7_pos,y_los_7_prev_%,y_mort_12mo_pos,y_mort_12mo_prev_%,y_icu_readmit_30_pos,y_icu_readmit_30_prev_%
str,u32,i64,f64,i64,f64,i64,f64,i64,f64
"""train""",49080,5004,10.195599,6678,13.606357,8271,16.852078,2093,4.264466
"""held_out""",6097,621,10.185337,763,12.514351,1029,16.877153,273,4.477612
"""tuning""",5986,642,10.725025,840,14.032743,1029,17.19011,227,3.792182


In [21]:
from datasets import load_from_disk
from tqdm.notebook import tqdm

In [22]:
a = load_from_disk('../data/desc_gen_dataset/')

Loading dataset from disk:   0%|          | 0/311 [00:00<?, ?it/s]

In [23]:
# import numpy as np

# event_counts = []
# word_counts = []

# for sample in tqdm(a["within24_genhpf"]):
#     event_counts.append(len(sample))
#     for event in sample:
#         word_counts.append(len(event.split()))

# print("mean events:", np.mean(event_counts))
# print("95th percentile events:", np.percentile(event_counts, 95))
# print("mean words/event:", np.mean(word_counts))
# print("95th percentile words/event:", np.percentile(word_counts, 95))

In [24]:


import os
import yaml
import json
import torch
import wandb
import polars as pl
from tqdm import tqdm
import torch.nn as nn
import torch.distributed as dist

from typing import Callable
from transformers import BertConfig
from torchmetrics.classification import BinaryAUROC, BinaryAveragePrecision
from transformers import CONFIG_MAPPING, MODEL_FOR_MASKED_LM_MAPPING, MODEL_MAPPING, MODEL_FOR_CAUSAL_LM_MAPPING

def correct_tokenizer_dict(dict_fp:str):
    with open(dict_fp) as f:
        old_dictionary = json.load(f)
    dictionary = {"age_stats": old_dictionary["age_stats"],
                  "is_hierarchical": old_dictionary.get("is_hierarchical", False),
                  "vocab": old_dictionary["regular"]}
    return dictionary

BERT_VARIANTS = {
    "bert": {},
    "medbert": dict(
        hidden_size=192,
        intermediate_size=64,
        num_attention_heads=6,
        num_hidden_layers=6,
        hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1,
    ),

    "cehrbert": dict(
        hidden_size=128,
        intermediate_size=2048,
        num_hidden_layers=12,
        num_attention_heads=8,
        hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1,
    ),

    "behrt": dict(
        hidden_size=288,
        intermediate_size=512,
        num_attention_heads=12,
        num_hidden_layers=6,
        hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1,
    ),

    "hibehrt": dict(
        hidden_size=150,
        intermediate_size=108,
        num_attention_heads=6,
        num_hidden_layers=4,
        hidden_dropout_prob=0.2,
        attention_probs_dropout_prob=0.3,
    ),
}



def get_config_and_model_cls(model_type: str, mode: str = "mlm", variant: str = None):

    assert mode in ["mlm", "eval", "causal"]

    if model_type not in CONFIG_MAPPING:
        raise ValueError(f"Unknown model_type: {model_type}")

    config_cls = CONFIG_MAPPING[model_type]

    if mode == "mlm":
        model_cls = MODEL_FOR_MASKED_LM_MAPPING[config_cls]
    elif mode == "eval":
        model_cls = MODEL_MAPPING[config_cls]
    else:
        model_cls = MODEL_FOR_CAUSAL_LM_MAPPING[config_cls]


    variant_kwargs = {}

    if variant is not None and issubclass(config_cls, BertConfig):
        if variant not in BERT_VARIANTS:
            raise ValueError(f"Unknown BERT variant: {variant}")

        variant_kwargs = BERT_VARIANTS[variant]


    def build_config(**kwargs):
        return config_cls(**variant_kwargs, **kwargs)

    return build_config, model_cls


def fix_roberta_longformer_max_pos(cfg):

    model_type = getattr(cfg, "model_type", "").lower()

    if model_type == "roberta":
        if cfg.max_position_embeddings == 512:
            cfg.max_position_embeddings = 513

    elif model_type == "longformer":
        if cfg.max_position_embeddings == 512:
            cfg.max_position_embeddings = 4097

    return cfg



def load_config_with_env(path):
    # read file
    with open(path, "r") as f:
        raw_text = f.read()
    expanded = os.path.expandvars(raw_text)
    
    return yaml.safe_load(expanded)


class Time2Vec(nn.Module):

    def __init__(
        self,
        in_features: int = 1,
        out_features: int = 16,
        periodic_activation: Callable = torch.sin,
    ):
        super().__init__()
        assert out_features >= 1, "out_features must be >= 1"

        self.in_features = in_features
        self.out_features = out_features
        self.periodic_activation = periodic_activation

        self.W = nn.Parameter(torch.randn(in_features, out_features - 1))
        self.b = nn.Parameter(torch.randn(out_features - 1))

        self.W0 = nn.Parameter(torch.randn(in_features))
        self.b0 = nn.Parameter(torch.randn(1))

    def forward(self, tau: torch.Tensor) -> torch.Tensor:

        v1 = self.periodic_activation(tau @ self.W + self.b)
        v2 = (tau @ self.W0).unsqueeze(-1) + self.b0

        return torch.cat([v2, v1], dim=-1)
    

def get_rank():
    if not dist.is_available() or not dist.is_initialized():
        return 0
    return dist.get_rank()




def get_bootstrap_ci(
    y_true: torch.Tensor,
    y_score: torch.Tensor,
    num_iter: int = 1000,
    alpha: float = 0.05,
    ndigits: int = 3,
):
    device = y_score.device

    y_true = y_true.detach().view(-1).to(device).long()
    y_score = y_score.detach().view(-1).to(device)

    auroc_point = BinaryAUROC().to(device)(y_score, y_true)
    auprc_point = BinaryAveragePrecision().to(device)(y_score, y_true)

    n = y_true.numel()
    auroc_samples = torch.empty(num_iter, device=device)
    auprc_samples = torch.empty(num_iter, device=device)

    for i in range(num_iter):
        idx = torch.randint(0, n, (n,), device=device)
        auroc_samples[i] = BinaryAUROC().to(device)(y_score[idx], y_true[idx])
        auprc_samples[i] = BinaryAveragePrecision().to(device)(y_score[idx], y_true[idx])

    # Percentile CI
    q_low = alpha / 2.0         # 2.5%
    q_high = 1.0 - alpha / 2.0  # 97.5%

    auroc_low = torch.quantile(auroc_samples, q_low)
    auroc_high = torch.quantile(auroc_samples, q_high)

    auprc_low = torch.quantile(auprc_samples, q_low)
    auprc_high = torch.quantile(auprc_samples, q_high)

    def _fmt(point, low, high):
        p = float(point.detach().cpu())
        l = float(low.detach().cpu())
        h = float(high.detach().cpu())
        return f"{round(p, ndigits)} ({round(l, ndigits)}, {round(h, ndigits)})"

    auroc_text = _fmt(auroc_point, auroc_low, auroc_high)
    auprc_text = _fmt(auprc_point, auprc_low, auprc_high)

    return auroc_text, auprc_text


def gather_1d_varlen_pl(module, x: torch.Tensor) -> torch.Tensor:
    x = x.detach().view(-1)

    if not getattr(module, "trainer", None) or module.trainer.world_size == 1:
        return x

    device = x.device
    local_len = torch.tensor([x.numel()], device=device, dtype=torch.long)

    all_lens = module.all_gather(local_len).view(-1) 
    max_len = int(all_lens.max().item())

    if x.numel() < max_len:
        pad = torch.zeros(max_len - x.numel(), device=device, dtype=x.dtype)
        x_pad = torch.cat([x, pad], dim=0)
    else:
        x_pad = x

    x_gather = module.all_gather(x_pad)

    chunks = []
    for r in range(x_gather.shape[0]):
        chunks.append(x_gather[r, : int(all_lens[r].item())])
    return torch.cat(chunks, dim=0)


def log_bootstrap_ci_text_percentile(
    module,
    y_true: torch.Tensor,
    y_score: torch.Tensor,
    prefix: str = "test",
    num_iter: int = 1000,
    alpha: float = 0.05,
    ndigits: int = 3,
):
    y_all = gather_1d_varlen_pl(module, y_true)
    s_all = gather_1d_varlen_pl(module, y_score)

    if not getattr(module, "trainer", None) or module.trainer.is_global_zero:
        auroc_ci_text, auprc_ci_text = get_bootstrap_ci(
            y_true=y_all,
            y_score=s_all,
            num_iter=num_iter,
            alpha=alpha,
            ndigits=ndigits,
        )

        # Use wandb.log directly for string-based CI values
        wandb.log({
            f"{prefix}_auroc_ci": auroc_ci_text,
            f"{prefix}_auprc_ci": auprc_ci_text,
        }, commit=False)



#########################################################
# LLM baselines
#########################################################

TASK_PROMPTS = {
    "y_los_7": (
        "You are an expert clinical risk prediction model using electronic health records.\n\n"
        "--- PATIENT DATA ---\n"
        "Electronic Health Records:\n"
        "{ehr_text}\n\n"
        "--- TASK ---\n"
        "will this patient have a hospital length of stay longer than 7 days?\n\n"
        "Answer only using one word: Yes or No.\n\n"
        "Answer:"
    ),
    "y_icu_readmit_30": (
        "You are an expert clinical risk prediction model using electronic health records.\n\n"
        "--- PATIENT DATA ---\n"
        "Electronic Health Records:\n"
        "{ehr_text}\n\n"
        "--- TASK ---\n"
        "Will this patient be readmitted to the ICU within 30 days after ICU discharge?\n\n"
        "Answer only using one word: Yes or No.\n\n"
        "Answer:"
    ),
    "y_mort": (
        "You are an expert clinical risk prediction model using electronic health records.\n\n"
        "--- PATIENT DATA ---\n"
        "Electronic Health Records:\n"
        "{ehr_text}\n\n"
        "--- TASK ---\n"
        "Will this patient die during this hospital admission?\n\n"
        "Answer only using one word: Yes or No.\n\n"
        "Answer:"
    ),
    "y_mort_12mo": (
        "You are an expert clinical risk prediction model using electronic health records.\n\n"
        "--- PATIENT DATA ---\n"
        "Electronic Health Records:\n"
        "{ehr_text}\n\n"
        "--- TASK ---\n"
        "Will this patient die within 1 year after hospital discharge?\n\n"
        "Answer only using one word: Yes or No.\n\n"
        "Answer:"
    ),
}


def build_prediction_prompt(ehr_text: str, task_name: str) -> str:
    if task_name not in TASK_PROMPTS:
        raise ValueError(f"Unknown task_name: {task_name}")
    return TASK_PROMPTS[task_name].format(ehr_text=ehr_text)


def predict_yes_no_probability(ehr_text: str, task_name: str, model_bundle: dict):
    model = model_bundle["model"]
    tokenizer = model_bundle["tokenizer"]
    yes_token_id = model_bundle["yes_token_id"]
    no_token_id = model_bundle["no_token_id"]

    prompt = build_prediction_prompt(ehr_text, task_name)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    next_token_logits = outputs.logits[:, -1, :].float()

    yes_logit = next_token_logits[0, yes_token_id]
    no_logit = next_token_logits[0, no_token_id]

    yes_no_logits = torch.stack([no_logit, yes_logit], dim=0)
    probs = torch.softmax(yes_no_logits, dim=0)

    no_prob = float(probs[0].cpu())
    yes_prob = float(probs[1].cpu())

    pred_label = 1 if yes_prob > 0.5 else 0
    pred_text = "Yes" if pred_label == 1 else "No"

    return {
        "prompt": prompt,
        "pred_text": pred_text,
        "pred_label": pred_label,
        "yes_prob": yes_prob,
        "no_prob": no_prob,
    }

def compute_metrics_with_ci_llm(results):

    y_true = torch.tensor([r["ground_truth"] for r in results], dtype=torch.long)
    y_score = torch.tensor([r["yes_prob"] for r in results], dtype=torch.float32)


    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    y_true = y_true.to(device)
    y_score = y_score.to(device)


    auroc = BinaryAUROC().to(device)(y_score, y_true)
    auprc = BinaryAveragePrecision().to(device)(y_score, y_true)


    auroc_ci, auprc_ci = get_bootstrap_ci(y_true, y_score)

    return {
        "AUROC": float(auroc.cpu()),
        "AUPRC": float(auprc.cpu()),
        "AUROC_CI": auroc_ci,
        "AUPRC_CI": auprc_ci,
    }


def predict_dataset(dataset, data_idx_path, window, task_name, model_bundle, split):

    results = []

    idx = pl.read_parquet(data_idx_path)
    test_idx = idx.filter(pl.col("split") == split)

    test_rows = {
        int(row["icustay_id"]): row
        for row in test_idx.to_dicts()
    }

    # build direct lookup once using only the icustay_id column
    icustay_ids = dataset["icustay_id"]

    hf_lookup = {
        int(stay_id): i
        for i, stay_id in enumerate(icustay_ids)
    }

    for stay_id, row in tqdm(test_rows.items()):

        hf_idx = hf_lookup[stay_id]
        sample = dataset[hf_idx]

        query = sample[window]

        prediction = predict_yes_no_probability(
            ehr_text="\n".join(query),
            task_name=task_name,
            model_bundle=model_bundle,
        )

        results.append({
            "subject_id": sample["subject_id"],
            "stay_id": stay_id,
            "ground_truth": row[task_name],
            "yes_prob": prediction["yes_prob"],
            "no_prob": prediction["no_prob"],
        })

    return results

In [25]:
import torch
import random
import time
import numpy as np
import polars as pl
from datasets import load_from_disk
from torch.utils.data import Dataset
from transformers import AutoTokenizer
from typing import Any, Dict, List, Optional, Tuple

In [26]:
class DescEmbDataset(Dataset):
    def __init__(
        self,
        dataset_path: str,
        data_idx_path: str,
        task: str = "y_mort",
        main_window: str = "within48_descemb",  
        split: str = "train",
        max_word_len: int = 32,                 
        max_events: int = None,                 
    ) -> None:

        self.task = task
        self.main_window = main_window
        self.max_word_len = max_word_len
        self.max_events = max_events
        

        self.data_idx = pl.scan_parquet(data_idx_path).collect()
        self.data_idx = self.data_idx.filter(pl.col("split") == split)
        self.data_idx = self.data_idx.filter(~pl.col("subject_id").is_in([15409850,16816440,18757959]) )


        self.hf_dataset = load_from_disk(dataset_path)

        subj_ids = self.hf_dataset["subject_id"]
        icu_ids = self.hf_dataset["icustay_id"]
        self._hf_index = {
            (int(s), int(i)): idx for idx, (s, i) in enumerate(zip(subj_ids, icu_ids))
        }


        self.tokenizer = AutoTokenizer.from_pretrained(
            "emilyalsentzer/Bio_ClinicalBERT"
        )

    def __len__(self) -> int:
        return len(self.data_idx)

    def __getitem__(self, idx: int):
        row = self.data_idx.row(idx, named=True)
        sid = int(row["subject_id"])
        hadm = int(row["hadm_id"])
        icu = int(row["icustay_id"])
        y = row[self.task]

        hf_idx = self._hf_index.get((sid, icu))
        if hf_idx is None:
            return None

        ex = self.hf_dataset[hf_idx]          
        events = ex[self.main_window] or []
        events = [e for e in events if isinstance(e, str) and e.strip()]
        if self.max_events is not None:
            events = events[: self.max_events]
        if len(events) == 0:
            return None

        enc = self.tokenizer(
            events,
            padding="max_length",
            truncation=True,
            max_length=self.max_word_len,
            add_special_tokens=True,
            return_tensors="pt",
        )

        return {
            "subject_id": sid,
            "hadm_id": hadm,
            "icustay_id": icu,
            "input_ids": enc["input_ids"].long(),
            "attention_mask": enc["attention_mask"].long(),
            "seq_len": torch.tensor(len(events), dtype=torch.long),
            "label": torch.tensor(float(y), dtype=torch.float32),
        }

In [27]:
class DescEmbCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, batch):


        batch = [b for b in batch if b is not None]
        if len(batch) == 0:
            return {}

        lengths = [int(b["seq_len"]) for b in batch]
        max_S = max(lengths)
        W = batch[0]["input_ids"].shape[1]
        B = len(batch)

        input_ids = torch.full((B, max_S, W), self.pad_token_id, dtype=torch.long)
        attention_mask = torch.zeros((B, max_S, W), dtype=torch.long)
        seq_len = torch.tensor(lengths, dtype=torch.long)
        labels = torch.stack([b["label"] for b in batch])

        for i, b in enumerate(batch):
            S_i = b["input_ids"].shape[0]
            input_ids[i, :S_i] = b["input_ids"]
            attention_mask[i, :S_i] = b["attention_mask"]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "seq_len": seq_len,
            "label": labels,

            "subject_id": torch.tensor(
                [b["subject_id"] for b in batch],
                dtype=torch.long,
            ),

            "hadm_id": torch.tensor(
                [b["hadm_id"] for b in batch],
                dtype=torch.long,
            ),

            "icustay_id": torch.tensor(
                [b["icustay_id"] for b in batch],
                dtype=torch.long,
            ),
        }

In [28]:

import math
import copy
import torch
import pandas as pd
import torch.nn as nn
import lightning.pytorch as lt
import torch.nn.functional as F

# from .models import EHREmbeddings
from torchmetrics import Accuracy
from typing import Optional, Tuple
from transformers import AutoModel, AutoConfig
# from .utils import log_bootstrap_ci_text_percentile
from transformers.modeling_outputs import BaseModelOutput
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torchmetrics.classification import BinaryAUROC, BinaryAveragePrecision
from transformers.models.roformer.modeling_roformer import RoFormerConfig, RoFormerEncoder

In [29]:
class BertEventEncoder(nn.Module):
    def __init__(
        self,
        bert_model_name: str = "emilyalsentzer/Bio_ClinicalBERT",
        pred_embed_dim: int = 128,
        init_bert_random: bool = False,
    ):
        super().__init__()

        if init_bert_random:
            config = AutoConfig.from_pretrained(bert_model_name)
            self.bert = AutoModel.from_config(config)
        else:
            self.bert = AutoModel.from_pretrained(bert_model_name)

        hidden_size = self.bert.config.hidden_size
        self.post_encode_proj = nn.Linear(hidden_size, pred_embed_dim)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        B, S, W = input_ids.shape

        flat_ids = input_ids.view(B * S, W)
        flat_mask = attention_mask.view(B * S, W)

        outputs = self.bert(
            input_ids=flat_ids,
            attention_mask=flat_mask,
        )
        cls_emb = outputs.last_hidden_state[:, 0, :] 

        event_emb = self.post_encode_proj(cls_emb)  
        event_emb = event_emb.view(B, S, -1)        
        return event_emb

In [30]:
class GRUEventHead(nn.Module):

    def __init__(self, pred_embed_dim: int=128, 
                 pred_hidden_dim: int=256, 
                 max_event_len: int =511,
                 n_layers: int = 1, 
                 dropout: float = 0.3, 
                 task: str = "binary"):
        super().__init__()
        self.pred_embed_dim = pred_embed_dim
        self.pred_hidden_dim = pred_hidden_dim
        self.n_layers = n_layers
        self.max_event_len = max_event_len
        self.task = task

        self.model = nn.GRU(
            input_size=self.pred_embed_dim,
            hidden_size=self.pred_hidden_dim,
            dropout=dropout if n_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=False,
            num_layers=self.n_layers,
        )

        out_dim = 18 if task == "diagnosis" else 1
        self.final_proj = nn.Linear(self.pred_hidden_dim, out_dim)

    def pack_pad_seq(self, x: torch.Tensor, lengths: torch.Tensor):
        lengths = lengths.view(-1).cpu()
        lengths[lengths > self.max_event_len] = self.max_event_len

        packed = pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False
        )
        output, _ = self.model(packed)
        output_seq, output_len = pad_packed_sequence(
            output, batch_first=True, padding_value=0.0
        )
        return output_seq, output_len

    def forward(self, x: torch.Tensor, seq_len: torch.Tensor) -> torch.Tensor:
       
        self.model.flatten_parameters()

        output_seq, _ = self.pack_pad_seq(x, seq_len) 
        i = range(x.size(0))
        last_hidden = output_seq[i, -1, :]             

        logits = self.final_proj(last_hidden)          
        if logits.shape[-1] == 1:
            logits = logits.squeeze(-1)                
        return logits

In [31]:
class DescEmbEvalModel(lt.LightningModule):
    def __init__(
        self,
        config,
        lr: float = 2e-5,
        wd: float = 0.0,
        max_epochs: int = 100,
        dropout: float = 0.1,
        freeze: bool = False,
        prediction_csv_path: str = '.'
    ):
        super().__init__()
        self.save_hyperparameters()

        bert_model_name = getattr(config, "bert_model_name", "google/bert_uncased_L-2_H-128_A-2")
        pred_embed_dim = getattr(config, "pred_embed_dim", 128)
        pred_hidden_dim = getattr(config, "pred_hidden_dim", 256)
        max_event_len = getattr(config, "max_event_len", 511)
        task = getattr(config, "task", "binary")

        self.encoder = BertEventEncoder(
            bert_model_name=bert_model_name,
            pred_embed_dim=pred_embed_dim,
            init_bert_random=getattr(config, "init_bert_random", False),
        )
        self.classifier = GRUEventHead(
            pred_embed_dim=pred_embed_dim,
            pred_hidden_dim=pred_hidden_dim,
            max_event_len=max_event_len,
            n_layers=getattr(config, "rnn_layer", 1),
            dropout=dropout,
            task=task,
        )

        if freeze:
            for p in self.encoder.parameters():
                p.requires_grad = False
            for p in self.classifier.parameters():
                p.requires_grad = True

        self.lr = lr
        self.wd = wd
        self.max_epochs = max_epochs

        self.criterion = nn.BCEWithLogitsLoss()
        self.prediction_csv_path = prediction_csv_path
        self.train_step_preds = []
        self.train_step_labels = []
        self.val_step_preds = []
        self.val_step_labels = []
        self.test_step_preds = []
        self.test_step_logits = [] 
        self.test_step_labels = []
        self.test_step_subject_ids = []
        self.test_step_hadm_ids = []
        self.test_step_icustay_ids = []

        self.train_auroc = BinaryAUROC()
        self.train_auprc = BinaryAveragePrecision()
        self.val_auroc = BinaryAUROC()
        self.val_auprc = BinaryAveragePrecision()
        self.test_auroc = BinaryAUROC()
        self.test_auprc = BinaryAveragePrecision()

    def forward(self, input_ids, attention_mask, seq_len=None, labels=None):
        event_emb = self.encoder(input_ids=input_ids, attention_mask=attention_mask)  

        if seq_len is None:
            event_mask = attention_mask.any(dim=-1)     
            seq_len = event_mask.sum(dim=-1)             
        logits = self.classifier(event_emb, seq_len)       
        return logits

    def training_step(self, batch, batch_idx):
        logits = self.forward(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            seq_len=batch["seq_len"],
        )

        y = batch["label"].float().view(-1)
        loss = self.criterion(logits, y)

        pos_score = torch.sigmoid(logits)

        self.train_step_labels.append(y.detach())
        self.train_step_preds.append(pos_score.detach())

        self.log("train_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_train_epoch_end(self) -> None:
        if len(self.train_step_labels) == 0:
            return

        y = torch.cat(self.train_step_labels)
        pos_score = torch.cat(self.train_step_preds)

        gathered_y = self.all_gather(y).reshape(-1)
        gathered_pos_score = self.all_gather(pos_score).reshape(-1)

        auroc = self.train_auroc(gathered_pos_score, gathered_y.long())
        auprc = self.train_auprc(gathered_pos_score, gathered_y.long())

        self.log("train_auroc", auroc, on_epoch=True, logger=True, prog_bar=True, sync_dist=True)
        self.log("train_auprc", auprc, on_epoch=True, logger=True, prog_bar=True, sync_dist=True)

        self.train_step_labels.clear()
        self.train_step_preds.clear()

    def validation_step(self, batch, batch_idx):
        logits = self.forward(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            seq_len=batch["seq_len"],
        )

        y = batch["label"].float().view(-1)
        loss = self.criterion(logits, y)
        pos_score = torch.sigmoid(logits)

        self.val_step_labels.append(y.detach())
        self.val_step_preds.append(pos_score.detach())

        self.log("val_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_validation_epoch_end(self, *args, **kwargs) -> None:
        if len(self.val_step_labels) == 0:
            return

        y = torch.cat(self.val_step_labels)
        pos_score = torch.cat(self.val_step_preds)

        gathered_y = self.all_gather(y).reshape(-1)
        gathered_pos_score = self.all_gather(pos_score).reshape(-1)

        auroc = self.val_auroc(gathered_pos_score, gathered_y.long())
        auprc = self.val_auprc(gathered_pos_score, gathered_y.long())

        self.log("val_auroc", auroc, on_epoch=True, logger=True, prog_bar=True, sync_dist=True)
        self.log("val_auprc", auprc, on_epoch=True, logger=True, prog_bar=True, sync_dist=True)

        self.val_step_labels.clear()
        self.val_step_preds.clear()

    def test_step(self, batch, batch_idx):
        logits = self.forward(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            seq_len=batch["seq_len"],
        )

        y = batch["label"].float().view(-1)
        loss = self.criterion(logits, y)
        pos_score = torch.sigmoid(logits)

        self.test_step_logits.append(logits.detach())
        self.test_step_preds.append(pos_score.detach())
        self.test_step_labels.append(y.detach())

        self.test_step_subject_ids.append(batch["subject_id"])
        self.test_step_hadm_ids.append(batch["hadm_id"])
        self.test_step_icustay_ids.append(batch["icustay_id"])

        self.log("test_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss


    def on_test_epoch_end(self,*arg, **kwargs) -> None:
        if len(self.test_step_labels) == 0:
            return
        y = torch.cat(self.test_step_labels)
        pos_score = torch.cat(self.test_step_preds)
        logits = torch.cat(self.test_step_logits)
        subject_ids = torch.cat(self.test_step_subject_ids)
        hadm_ids = torch.cat(self.test_step_hadm_ids)
        icustay_ids = torch.cat(self.test_step_icustay_ids)

        gathered_y = self.all_gather(y)
        gathered_pos_score = self.all_gather(pos_score)
        gathered_logits = self.all_gather(logits)

        gathered_subject_ids = self.all_gather(subject_ids)
        gathered_hadm_ids = self.all_gather(hadm_ids)
        gathered_icustay_ids = self.all_gather(icustay_ids)

        gathered_y = gathered_y.reshape(-1)
        gathered_pos_score = gathered_pos_score.reshape(-1)
        gathered_logits = gathered_logits.reshape(-1)

        gathered_subject_ids = gathered_subject_ids.reshape(-1)
        gathered_hadm_ids = gathered_hadm_ids.reshape(-1)
        gathered_icustay_ids = gathered_icustay_ids.reshape(-1)

        auroc = self.test_auroc(gathered_pos_score,gathered_y.long())
        auprc = self.test_auprc(gathered_pos_score,gathered_y.long())

        self.log("test_auroc", auroc, on_epoch=True, logger=True,)
        self.log("test_auprc", auprc, on_epoch=True,logger=True)

        log_bootstrap_ci_text_percentile(
            module=self,
            y_true=gathered_y,
            y_score=gathered_pos_score,
            prefix="test",
            num_iter=1000,
            alpha=0.05,
            ndigits=3)

        if self.global_rank == 0:

            results = pd.DataFrame(
                {
                    "subject_id": gathered_subject_ids.cpu().numpy(),
                    "hadm_id": gathered_hadm_ids.cpu().numpy(),
                    "icustay_id": gathered_icustay_ids.cpu().numpy(),
                    "label": gathered_y.cpu().numpy(),
                    "prediction": gathered_pos_score.cpu().numpy(),
                    "logit": gathered_logits.cpu().numpy(),
                }
            )

            results.to_csv(self.prediction_csv_path, index=False)

        self.test_step_labels.clear()
        self.test_step_preds.clear()
        self.test_step_logits.clear()
        self.test_step_subject_ids.clear()
        self.test_step_hadm_ids.clear()
        self.test_step_icustay_ids.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.wd)
        # scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        #     optimizer=optimizer,
        #     eta_min=0,
        #     T_max=self.max_epochs,
        # )
        return {"optimizer": optimizer}#, "lr_scheduler": scheduler}

In [32]:
from types import SimpleNamespace
prediction_csv_path = '.'
train_dataset = DescEmbDataset(dataset_path='../data/desc_gen_dataset/',
                                            data_idx_path='../resources/downstream_index.parquet',
                                            task='y_mort',
                                            main_window='within48_descemb',
                                            max_word_len=12,
                                            max_events=510,
                                            split='train')

val_dataset = DescEmbDataset(dataset_path='../data/desc_gen_dataset/',
                                            data_idx_path='../resources/downstream_index.parquet',
                                            task='y_mort',
                                            main_window='within48_descemb',
                                            max_word_len=12,
                                            max_events=510,
                                            split='tuning')
collate_fn = DescEmbCollator(pad_token_id=train_dataset.tokenizer.pad_token_id)
cfg = SimpleNamespace(bert_model_name="google/bert_uncased_L-2_H-128_A-2",
                    pred_embed_dim=128,
                    pred_hidden_dim=256,     
                    max_event_len=510,       
                    rnn_layer=1,
                    init_bert_random=False,  
                    task="binary")
model = DescEmbEvalModel(config=cfg,
                        lr=1e-5,
                        max_epochs=75,
                        dropout=0.1,
                        freeze=True,
                        prediction_csv_path=prediction_csv_path)

Loading dataset from disk:   0%|          | 0/311 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [46]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(dataset=train_dataset,
                            batch_size=64,
                            collate_fn=collate_fn,
                            num_workers=14,
                            shuffle=True,
                            pin_memory=True,
                            persistent_workers=True,
                            prefetch_factor=4
                             )
val_dataloader = DataLoader(dataset=val_dataset,
                            batch_size=64,
                            collate_fn=collate_fn,
                            num_workers=24,
                            shuffle=False,
                            pin_memory=True,
                            persistent_workers=True,
                            prefetch_factor=4)

In [47]:
trainer = lt.Trainer(accelerator='auto', 
                    devices='auto',
                    strategy='auto',
#                   logger=wandb_logger, 
                    log_every_n_steps=1,
                    num_sanity_val_steps=0,
                    max_epochs=75,
                    precision="bf16-mixed",
#                   callbacks=[early_stop,lr_monitor, checkpoint_callback],
#                   enable_checkpointing=True
                    )

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [48]:
# trainer.fit(model=model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

In [49]:
loader_iter = iter(train_dataloader)

for i in range(25):
    t0 = time.time()

    batch = next(loader_iter)

    elapsed = time.time() - t0

    print(
        f"batch {i}: "
        f"load={elapsed:.3f}s "
        f"shape={batch['input_ids'].shape} "
        f"max_events={batch['seq_len'].max().item()}"
    )

batch 0: load=13.760s shape=torch.Size([64, 510, 12]) max_events=510
batch 1: load=0.001s shape=torch.Size([64, 510, 12]) max_events=510
batch 2: load=0.313s shape=torch.Size([64, 510, 12]) max_events=510
batch 3: load=0.000s shape=torch.Size([64, 510, 12]) max_events=510
batch 4: load=0.000s shape=torch.Size([64, 510, 12]) max_events=510
batch 5: load=0.450s shape=torch.Size([64, 510, 12]) max_events=510
batch 6: load=0.000s shape=torch.Size([64, 510, 12]) max_events=510
batch 7: load=0.000s shape=torch.Size([64, 510, 12]) max_events=510
batch 8: load=0.000s shape=torch.Size([64, 510, 12]) max_events=510
batch 9: load=0.000s shape=torch.Size([64, 510, 12]) max_events=510
batch 10: load=0.000s shape=torch.Size([64, 510, 12]) max_events=510
batch 11: load=0.000s shape=torch.Size([64, 510, 12]) max_events=510
batch 12: load=0.000s shape=torch.Size([64, 510, 12]) max_events=510
batch 13: load=0.000s shape=torch.Size([64, 510, 12]) max_events=510
batch 14: load=13.658s shape=torch.Size([64

In [52]:
t0=time.time()
batch = collate_fn([train_dataset[i] for i in range(64)])
print(time.time()-t0)

9.419044971466064
